# `build_sector_db` — SQLite Usage Guide

Hands-on guide to building and querying `data/sector_results.db` — the
normalized SQLite mirror of all daily sector-analysis JSON files.

**No API key required.** Everything runs on stdlib `sqlite3` and `pandas`.

```
build_sector_db.py
│
└── Public API
    └── build(db_path, results_dir) -> int   full rebuild, atomic write
```

**Schema at a glance:**

```
sector_analyses  (id, date, sector, sentiment, sentiment_score,
                  news_category, extraction_status, batch_id)
       │
       └─► sector_entities  (id, analysis_id FK, entity)
```

## Sections
1. [Setup](#1-setup)
2. [Build / rebuild](#2-build--rebuild)
3. [Schema reference](#3-schema-reference)
4. [Querying `sector_analyses`](#4-querying-sector_analyses)
5. [Entity lookups](#5-entity-lookups)
6. [Pandas integration](#6-pandas-integration)
7. [End-to-end: sentiment trend for a sector](#7-end-to-end-sentiment-trend-for-a-sector)
8. [Error handling reference](#8-error-handling-reference)

---
## 1 — Setup

Standard-library and project imports. `SECTOR_DB_FILE` and
`SECTOR_RESULTS_DIR` are the canonical path constants; all sections below
use these instead of hard-coded strings so the notebook stays correct if
paths ever change.

In [ ]:
import sqlite3
import sys
from pathlib import Path

import pandas as pd

# Ensure the project root is on sys.path when running from the notebooks/ dir.
PROJECT_ROOT = Path(__file__).parent.parent if "__file__" in dir() else Path("..")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from build_sector_db import build
from constants import SECTOR_DB_FILE, SECTOR_RESULTS_DIR

print(f"DB path          : {SECTOR_DB_FILE}")
print(f"Results dir      : {SECTOR_RESULTS_DIR}")
print(f"Results dir exists: {SECTOR_RESULTS_DIR.exists()}")

---
## 2 — Build / rebuild

`build()` performs a **full rebuild** on every call:

1. Collects all `*.json` files in `results_dir`, sorted chronologically by filename.
2. Connects to a temporary `.db.tmp` file (never touches the live DB during the write).
3. Executes the DDL (creates tables + indices, idempotent).
4. Inserts every `SectorAnalysis` record and its associated entities.
5. `os.replace()` swaps the tmp file to the final path — atomic on all platforms.

Returns the number of `sector_analyses` rows inserted (one per sector × date).

> **Invariant:** `date` is taken from the **filename stem** (`2026-03-12.json`
> → `"2026-03-12"`), not from any field inside the JSON body. This keeps the
> DB consistent even if the JSON body contains a slightly different date string.

In [ ]:
row_count = build(db_path=SECTOR_DB_FILE, results_dir=SECTOR_RESULTS_DIR)

db_size_kb = SECTOR_DB_FILE.stat().st_size / 1024
json_count = sum(1 for _ in SECTOR_RESULTS_DIR.glob("*.json"))

print(f"sector_analyses rows : {row_count}")
print(f"JSON files processed : {json_count}")
print(f"DB size              : {db_size_kb:.1f} KB")

---
## 3 — Schema reference

Use `PRAGMA table_info` to inspect column names, types, nullability, and
default values at runtime. This is the authoritative source — it reflects
whatever schema is actually in the file, not documentation that might drift.

In [ ]:
conn = sqlite3.connect(SECTOR_DB_FILE)

for table in ("sector_analyses", "sector_entities"):
    cols = conn.execute(f"PRAGMA table_info({table})").fetchall()
    print(f"\n{table}")
    print(f"  {'cid':<4} {'name':<20} {'type':<10} {'notnull':<8} {'dflt'}")
    print(f"  {'-'*55}")
    for cid, name, typ, notnull, dflt, pk in cols:
        print(f"  {cid:<4} {name:<20} {typ:<10} {notnull:<8} {dflt}")

In [ ]:
# List all indices — useful to know which columns are fast to filter on.
indices = conn.execute("SELECT name, tbl_name, sql FROM sqlite_master WHERE type='index'").fetchall()
print(f"{'Index name':<30} {'Table':<20}")
print("-" * 50)
for name, tbl, sql in indices:
    print(f"{name:<30} {tbl:<20}")

---
## 4 — Querying `sector_analyses`

The `sector_analyses` table has one row per **date × sector** combination.
Common filter dimensions: `date`, `sector`, `sentiment`.

`sentiment_score` is pre-denormalized (`positive`→1, `neutral`→0,
`negative`→−1) so aggregate queries never need a `CASE` expression.

In [ ]:
# Most recent 5 rows — sanity check that the build succeeded.
rows = conn.execute("""
    SELECT date, sector, sentiment, sentiment_score, news_category
    FROM   sector_analyses
    ORDER  BY date DESC
    LIMIT  5
""").fetchall()

print(f"{'date':<12} {'sector':<25} {'sentiment':<10} {'score':>5}  {'news_category'}")
print("-" * 72)
for date, sector, sentiment, score, cat in rows:
    print(f"{date:<12} {sector:<25} {sentiment:<10} {score:>5}  {cat}")

In [ ]:
# Filter by sector — all entries for "Technology Services".
rows = conn.execute("""
    SELECT date, sentiment, sentiment_score, news_category
    FROM   sector_analyses
    WHERE  sector = 'Technology Services'
    ORDER  BY date DESC
    LIMIT  10
""").fetchall()

print(f"Technology Services — last {len(rows)} entries")
print(f"{'date':<12} {'sentiment':<12} {'score':>5}  {'news_category'}")
print("-" * 46)
for date, sentiment, score, cat in rows:
    print(f"{date:<12} {sentiment:<12} {score:>5}  {cat}")

In [ ]:
# Aggregate — mean sentiment_score per sector over all dates.
rows = conn.execute("""
    SELECT   sector,
             ROUND(AVG(sentiment_score), 3) AS mean_score,
             COUNT(*)                        AS n_days
    FROM     sector_analyses
    GROUP BY sector
    ORDER BY mean_score DESC
""").fetchall()

print(f"{'sector':<28} {'mean_score':>10}  {'n_days':>6}")
print("-" * 50)
for sector, mean_score, n in rows:
    bar = "#" * int((mean_score + 1) * 10)   # scale -1..+1 to 0..20
    print(f"{sector:<28} {mean_score:>10.3f}  {n:>6}  {bar}")

---
## 5 — Entity lookups

Entities are stored in the **normalized** `sector_entities` table — one row
per mention, with a foreign key back to `sector_analyses`. This replaces the
pipe-delimited string in `sector_summary.tsv` and makes entity queries fast
via the `lower(entity)` index.

**Pattern:** always use `lower(se.entity) = lower(?)` so the lookup is
case-insensitive regardless of how the LLM capitalised the name.

In [ ]:
# Recent appearances of an entity across all sectors.
ENTITY = "Nvidia"

rows = conn.execute("""
    SELECT sa.date, sa.sector, sa.sentiment, sa.sentiment_score, se.entity
    FROM   sector_analyses sa
    JOIN   sector_entities se ON se.analysis_id = sa.id
    WHERE  lower(se.entity) = lower(?)
    ORDER  BY sa.date DESC
    LIMIT  10
""", (ENTITY,)).fetchall()

print(f"Entity '{ENTITY}' — last {len(rows)} mentions")
print(f"{'date':<12} {'sector':<25} {'sentiment':<12} {'score':>5}")
print("-" * 58)
for date, sector, sentiment, score, entity in rows:
    print(f"{date:<12} {sector:<25} {sentiment:<12} {score:>5}")

In [ ]:
# Top 20 most frequently mentioned entities across all dates.
rows = conn.execute("""
    SELECT   lower(se.entity) AS entity_lower,
             se.entity        AS canonical,
             COUNT(*)         AS mentions
    FROM     sector_entities se
    GROUP BY entity_lower
    ORDER BY mentions DESC
    LIMIT    20
""").fetchall()

print(f"{'entity':<30} {'mentions':>8}")
print("-" * 40)
for entity_lower, canonical, mentions in rows:
    print(f"{canonical:<30} {mentions:>8}")

---
## 6 — Pandas integration

`pd.read_sql_query` turns any SQL result directly into a DataFrame.
Pass the open `sqlite3.Connection` as the `con` argument — no SQLAlchemy
engine required.

In [ ]:
# Load all sector_analyses into a DataFrame.
df = pd.read_sql_query(
    "SELECT * FROM sector_analyses ORDER BY date, sector",
    con=conn,
    parse_dates=["date"],
)

print(f"Shape    : {df.shape}")
print(f"Columns  : {list(df.columns)}")
print(f"Date range: {df['date'].min().date()} -> {df['date'].max().date()}")
df.head()

In [ ]:
# Pivot — wide table: date x sector, values = mean sentiment_score.
# Handy for heatmaps or correlation analysis.
pivot = (
    df.pivot_table(
        index="date",
        columns="sector",
        values="sentiment_score",
        aggfunc="mean",
    )
    .sort_index()
)

print(f"Pivot shape: {pivot.shape}  (dates x sectors)")
pivot.tail(3)

In [ ]:
# Join entities back to analyses — one row per entity mention.
df_entities = pd.read_sql_query(
    """
    SELECT sa.date, sa.sector, sa.sentiment_score, se.entity
    FROM   sector_analyses sa
    JOIN   sector_entities se ON se.analysis_id = sa.id
    ORDER  BY sa.date, sa.sector
    """,
    con=conn,
    parse_dates=["date"],
)

print(f"Shape: {df_entities.shape}")
df_entities.head()

---
## 7 — End-to-end: sentiment trend for a sector

Combines a parameterized SQL query, a rolling mean, and a matplotlib plot
to produce a sentiment trend chart for any sector — without reading any TSV.

Change `SECTOR` and `LOOKBACK_DAYS` to explore different sectors and windows.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

SECTOR       = "Technology Services"
LOOKBACK_DAYS = 90

# 1. Pull the time series from SQLite.
df_trend = pd.read_sql_query(
    """
    SELECT date, sentiment_score
    FROM   sector_analyses
    WHERE  sector = ?
      AND  date  >= date('now', ?)
    ORDER  BY date
    """,
    con=conn,
    params=(SECTOR, f"-{LOOKBACK_DAYS} days"),
    parse_dates=["date"],
)

if df_trend.empty:
    print(f"No data for '{SECTOR}' in the last {LOOKBACK_DAYS} days.")
else:
    # 2. Rolling 7-day mean to smooth daily noise.
    df_trend = df_trend.set_index("date").sort_index()
    df_trend["rolling_7d"] = (
        df_trend["sentiment_score"]
        .rolling("7D", min_periods=1)
        .mean()
    )

    # 3. Characterize trend (same threshold as query_sector.py).
    midpoint = len(df_trend) // 2
    first_half  = df_trend["sentiment_score"].iloc[:midpoint].mean()
    second_half = df_trend["sentiment_score"].iloc[midpoint:].mean()
    delta = second_half - first_half
    direction = (
        "improving"    if delta >  0.20 else
        "deteriorating" if delta < -0.20 else
        "stable"
    )

    print(f"Sector         : {SECTOR}")
    print(f"Lookback       : {LOOKBACK_DAYS} days")
    print(f"Observations   : {len(df_trend)}")
    print(f"Mean score     : {df_trend['sentiment_score'].mean():.3f}")
    print(f"Trend          : {direction}  (delta={delta:+.3f})")

    # 4. Plot.
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(
        df_trend.index, df_trend["sentiment_score"],
        color=[("#27ae60" if s > 0 else "#c0392b" if s < 0 else "#95a5a6")
               for s in df_trend["sentiment_score"]],
        alpha=0.5, label="Daily score",
    )
    ax.plot(df_trend.index, df_trend["rolling_7d"], color="#2c3e50",
            linewidth=2, label="7-day rolling mean")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
    plt.xticks(rotation=45, ha="right")
    ax.set_ylabel("Sentiment score  (-1 / 0 / +1)")
    ax.set_title(f"{SECTOR} — sentiment trend ({LOOKBACK_DAYS}-day window)")
    ax.legend()
    plt.tight_layout()
    plt.show()

---
## 8 — Error handling reference

### `build()` — edge cases

| Condition | Behaviour |
|---|---|
| `results_dir` is empty (no `*.json` files) | `build()` returns `0`; `main()` prints a warning to stderr and exits without writing a DB file |
| A JSON file is malformed | That file is skipped with a `stderr` log line; the build continues with remaining files |
| The process crashes mid-write | The `.db.tmp` temp file is left on disk; the next `build()` call removes it before starting |
| `db_path` parent directory does not exist | `sqlite3.connect` raises `OperationalError` — create the directory first |

### Opening a DB that does not exist yet

If you open a path that does not exist, SQLite creates a new **empty** file.
Always call `build()` first, or check existence before connecting.

In [ ]:
import tempfile

# Demonstrate: build() on an empty directory returns 0 rows.
with tempfile.TemporaryDirectory() as tmpdir:
    empty_results = Path(tmpdir) / "sector_results"
    empty_results.mkdir()
    empty_db = Path(tmpdir) / "empty.db"

    n = build(db_path=empty_db, results_dir=empty_results)
    print(f"Rows inserted from empty dir: {n}")
    # build() still creates the DB with schema but no data.
    if empty_db.exists():
        c = sqlite3.connect(empty_db)
        count = c.execute("SELECT COUNT(*) FROM sector_analyses").fetchone()[0]
        c.close()
        print(f"sector_analyses rows in empty DB: {count}")

In [ ]:
# Demonstrate: malformed JSON is skipped; build continues.
import json as _json

with tempfile.TemporaryDirectory() as tmpdir:
    results_dir = Path(tmpdir) / "sector_results"
    results_dir.mkdir()

    # One valid file.
    valid = {
        "batch_id": "batch_demo",
        "sectors": [
            {"sector": "Finance", "sentiment": "positive",
             "news_category": "earnings", "extraction_status": "ok",
             "entities": ["JPMorgan"]},
        ],
    }
    (results_dir / "2026-01-10.json").write_text(_json.dumps(valid))

    # One malformed file.
    (results_dir / "2026-01-11.json").write_text("{ this is not valid JSON !!!")

    db_path = Path(tmpdir) / "test.db"
    n = build(db_path=db_path, results_dir=results_dir)
    print(f"Rows inserted (1 valid + 1 bad file): {n}")  # expect 1

In [ ]:
# Close the shared connection opened in section 3.
conn.close()
print("Connection closed.")